# 5. GNN Training

This notebook trains a Graph Neural Network (HeteroSAGE) using the data processed in previous steps.
It loads the `HeteroData` object, sets up `NeighborLoader`, and executes the training loop.

**M1 Pro Optimized**: This version is configured to use the MPS backend and optimized DataLoader settings for Apple Silicon.

In [ ]:
# Configuration
import os
import torch

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
HETERO_GRAPH_PATH = os.path.join(ROOT_DIR, "processed_data", "hetero_graph.pt")
LABEL_CSV_PATH = os.path.join(ROOT_DIR, "data", "labeled_fraud.csv")
MODEL_SAVE_DIR = os.path.join(ROOT_DIR, "models")

# Training Hyperparameters
HIDDEN_CHANNELS = 64
NUM_LAYERS = 2
LEARNING_RATE = 0.001
EPOCHS = 10
BATCH_SIZE = 1024
NUM_NEIGHBORS = [10, 10]

# M1 Pro Optimization: Device Selection
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# M1 Pro Optimization: DataLoader settings
# Unified memory benefits from some parallel workers, but avoid too many spawn overheads.
NUM_WORKERS = 4 
PERSISTENT_WORKERS = True
# Pin memory speeds up host-to-device transfer (important for MPS)
PIN_MEMORY = True if DEVICE == 'mps' else False

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [ ]:
# Imports
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.transforms import ToUndirected
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import copy

print(f"Using device: {DEVICE}")

## 1. Load Data
Load the graph structure and the labels.

In [ ]:
print(f"Loading graph from {HETERO_GRAPH_PATH}...")
data = torch.load(HETERO_GRAPH_PATH, map_location="cpu", weights_only=False)
data = ToUndirected()(data)

# Initialize features if they are missing (Random features for demonstration if real features aren't loaded)
# In a real scenario, you should have loaded features in Step 2.
for node_type in data.node_types:
    if not hasattr(data[node_type], 'x') or data[node_type].x is None:
        print(f"Warning: Node type '{node_type}' has no features. Initializing random features.")
        data[node_type].x = torch.randn(data[node_type].num_nodes, 16)

# Load Labels
if os.path.exists(LABEL_CSV_PATH):
    print(f"Loading labels from {LABEL_CSV_PATH}...")
    df_label = pd.read_csv(LABEL_CSV_PATH)
    # Assume CSV has 'PN' (or id) and 'label'
    # We need to map 'PN' to the integer index of 'pekerja' node type used in PyG
    # NOTE: This requires the mapping created/used during Step 1 & 2. 
    # For this script we assume there is a mapping or IDs match. 
    # If IDs in CSV are raw strings/ints, they must be mapped to 0..N-1 range.
    
    # Example placeholder logic for mapping:
    # mapping = load_mapping("pekerja") 
    # df_label['nid'] = df_label['PN'].map(mapping)
    
    # If we assume 'nid' is already in dataframe or we just take top N nodes as labeled:
    data['pekerja'].y = torch.zeros(data['pekerja'].num_nodes, dtype=torch.float)
    # Setting dummy labels for demonstration if mapping isn't available yet
    # Replace this with actual mapping logic
    num_labeled = min(len(df_label), data['pekerja'].num_nodes)
    indices = torch.randperm(data['pekerja'].num_nodes)[:num_labeled]
    data['pekerja'].y[indices] = torch.randint(0, 2, (num_labeled,), dtype=torch.float)
    
    # Create Train/Val Split
    train_mask = torch.zeros(data['pekerja'].num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(data['pekerja'].num_nodes, dtype=torch.bool)
    
    train_idx = indices[:int(0.8*num_labeled)]
    val_idx = indices[int(0.8*num_labeled):]
    
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    
    data['pekerja'].train_mask = train_mask
    data['pekerja'].val_mask = val_mask
    
else:
    print("Label file not found. Generating dummy labels for testing code.")
    data['pekerja'].y = torch.randint(0, 2, (data['pekerja'].num_nodes,), dtype=torch.float)
    mask = torch.rand(data['pekerja'].num_nodes) < 0.8
    data['pekerja'].train_mask = mask
    data['pekerja'].val_mask = ~mask
    train_idx = torch.nonzero(mask, as_tuple=False).view(-1)
    val_idx = torch.nonzero(~mask, as_tuple=False).view(-1)

In [ ]:
# Create Loaders
train_loader = NeighborLoader(
    data,
    num_neighbors={key: NUM_NEIGHBORS for key in data.edge_types},
    input_nodes=('pekerja', train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    persistent_workers=PERSISTENT_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = NeighborLoader(
    data,
    num_neighbors={key: NUM_NEIGHBORS for key in data.edge_types},
    input_nodes=('pekerja', val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=PERSISTENT_WORKERS,
    pin_memory=PIN_MEMORY
)

## 2. Define Model
HeteroGraphSAGE architecture

In [ ]:
class HeteroSAGE(torch.nn.Module):
    def __init__(self, metadata, hidden_channels, out_channels, num_layers):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((-1, -1), hidden_channels)
                for edge_type in metadata[1]
            }, aggr='sum')
            self.convs.append(conv)

        self.lin = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: F.relu(x) for key, x in x_dict.items()}
            
        return self.lin(x_dict['pekerja'])

model = HeteroSAGE(
    metadata=data.metadata(),
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=1,
    num_layers=NUM_LAYERS
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 3. Training Loop

In [ ]:
def train():
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc="Training"):
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        out = model(batch.x_dict, batch.edge_index_dict)
        # Squeeze output to match labels [batch_size]
        out = out.squeeze()
        
        # Get labels for the batch
        # batch['pekerja'].y contains labels for all nodes in batch, initial seed nodes are first batch_size
        y = batch['pekerja'].y[:out.size(0)]
        
        loss = F.binary_cross_entropy_with_logits(out, y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * out.size(0)
        
    return total_loss / len(train_loader.dataset)

@torch.no_grad()
def test(loader):
    model.eval()
    preds, targets = [], []
    for batch in tqdm(loader, desc="Evaluating"):
        batch = batch.to(DEVICE)
        out = model(batch.x_dict, batch.edge_index_dict).squeeze()
        y = batch['pekerja'].y[:out.size(0)]
        
        preds.append(torch.sigmoid(out).cpu())
        targets.append(y.cpu())
        
    preds = torch.cat(preds)
    targets = torch.cat(targets)
    
    return roc_auc_score(targets, preds)

print("Starting training...")
for epoch in range(1, EPOCHS + 1):
    loss = train()
    auc = test(val_loader)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val AUC: {auc:.4f}')

# Save Model
save_path = os.path.join(MODEL_SAVE_DIR, "gnn_model.pth")
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")